# Gemma 4 뉴스 분석 LoRA -> GGUF 변환 Colab

`train_gemma4_colab.ipynb` 실행 결과로 생긴 `lora_adapter`를 base model(`google/gemma-4-E4B-it`)에 병합한 뒤, `llama.cpp`로 Ollama에서 사용할 `Q4_K_M.gguf` 파일을 만듭니다.

중요:
- Colab은 로컬 Mac 경로인 `/Volumes/PortableSSD/Downloads/gemma4_news_analyzer_outputs`를 직접 읽을 수 없습니다.
- Drive에 `/MyDrive/dev-jinro/outputs/gemma4_news_analyzer/lora_adapter`가 아직 남아 있으면 그대로 사용합니다.
- Drive에 없다면 Mac에서 `gemma4_news_analyzer_outputs.zip`을 만든 뒤 이 노트북의 업로드 셀에서 올리세요.
- 권장 런타임은 Colab L4/A100 GPU + High-RAM입니다. 변환 자체는 CPU 작업이지만, LoRA 병합 단계에서 base model을 로드해야 합니다.

Mac에서 zip을 만드는 예:

```bash
cd /Volumes/PortableSSD/Downloads
zip -r gemma4_news_analyzer_outputs.zip gemma4_news_analyzer_outputs
```


## 1. 패키지 설치

Gemma 4 지원은 라이브러리 버전에 민감하므로 Transformers v5 이상을 사용합니다. Colab 기본 이미지에 오래된 `torchao`가 있으면 PEFT가 adapter 로드 중 실패하므로 제거합니다.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=5.5.0" peft accelerate safetensors sentencepiece protobuf huggingface_hub packaging gguf

import importlib.metadata

try:
    print("torchao version:", importlib.metadata.version("torchao"))
except importlib.metadata.PackageNotFoundError:
    print("torchao is not installed; PEFT will skip the optional torchao backend.")


## 2. Drive, 인증, 작업 경로

HF 모델 접근이 막히면 Colab 왼쪽 `Secrets`에 `HF_TOKEN`을 추가하거나, 아래 셀의 `notebook_login()` 주석을 해제하세요.

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files, userdata
from huggingface_hub import login, notebook_login

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/dev-jinro")
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "gemma4_news_analyzer"
ADAPTER_DIR = OUTPUT_ROOT / "lora_adapter"
MERGED_DIR = OUTPUT_ROOT / "merged_16bit"
GGUF_DIR = OUTPUT_ROOT / "gguf_q4_k_m"
PACKAGE_DIR = OUTPUT_ROOT / "ollama_package"
UPLOAD_ROOT = Path("/content/uploaded_gemma4_news_analyzer_outputs")
LOCAL_ARCHIVE_HINT = "/Volumes/PortableSSD/Downloads/gemma4_news_analyzer_outputs.zip"

for directory in [DRIVE_ROOT, OUTPUT_ROOT, GGUF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    HF_TOKEN = userdata.get("HF_TOKEN") or os.getenv("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("HF_TOKEN이 없습니다. google/gemma-4-E4B-it 접근이 막히면 notebook_login() 주석을 해제하세요.")
    # notebook_login()

print("Drive root:", DRIVE_ROOT)
print("Expected adapter:", ADAPTER_DIR)
print("Local zip hint:", LOCAL_ARCHIVE_HINT)


## 3. LoRA adapter 준비

Drive에 adapter가 있으면 그대로 쓰고, 없으면 `gemma4_news_analyzer_outputs.zip`을 업로드해 Drive 작업공간으로 복사합니다.

In [ ]:
def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def find_adapter_root(search_root: Path) -> Path:
    candidates = sorted(search_root.rglob("lora_adapter/adapter_config.json"))
    if not candidates:
        raise FileNotFoundError(f"{search_root} 아래에서 lora_adapter/adapter_config.json을 찾지 못했습니다.")
    return candidates[0].parent


def ensure_adapter_dir() -> Path:
    if (ADAPTER_DIR / "adapter_config.json").exists():
        print("Using existing Drive adapter:", ADAPTER_DIR)
        return ADAPTER_DIR

    print("Drive에 LoRA adapter가 없습니다.")
    print("Mac에서 만든 zip을 업로드하세요:", LOCAL_ARCHIVE_HINT)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("업로드된 파일이 없습니다.")

    reset_dir(UPLOAD_ROOT)
    archive_names = [name for name in uploaded if name.endswith(".zip")]
    if not archive_names:
        raise ValueError("zip 파일을 업로드해야 합니다. 폴더 업로드 대신 gemma4_news_analyzer_outputs.zip을 사용하세요.")

    archive_path = Path("/content") / archive_names[0]
    shutil.unpack_archive(str(archive_path), str(UPLOAD_ROOT))
    uploaded_adapter_dir = find_adapter_root(UPLOAD_ROOT)
    uploaded_output_root = uploaded_adapter_dir.parent

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(uploaded_output_root, OUTPUT_ROOT, dirs_exist_ok=True)
    if not (ADAPTER_DIR / "adapter_config.json").exists():
        raise FileNotFoundError("업로드 압축 해제 후에도 Drive 작업공간에서 adapter_config.json을 찾지 못했습니다.")
    print("Uploaded adapter copied to Drive:", ADAPTER_DIR)
    return ADAPTER_DIR


adapter_dir = ensure_adapter_dir()
adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
MODEL_ID = adapter_config.get("base_model_name_or_path") or "google/gemma-4-E4B-it"

print("Base model:", MODEL_ID)
print("PEFT type:", adapter_config.get("peft_type"))
print("Target modules:", adapter_config.get("target_modules"))
print("Adapter files:")
for path in sorted(adapter_dir.iterdir()):
    if path.is_file():
        print("-", path.name, f"({path.stat().st_size / 1024**2:.2f} MB)")


## 4. Base model + LoRA 병합

PEFT adapter를 base model에 `merge_and_unload()`로 합친 뒤 표준 Hugging Face 모델 폴더(`merged_16bit`)로 저장합니다.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    torch_dtype = torch.bfloat16 if capability[0] >= 8 else torch.float16
    print("GPU:", torch.cuda.get_device_name(0), "dtype:", torch_dtype)
else:
    torch_dtype = torch.float16
    print("CUDA GPU가 없습니다. CPU 병합은 오래 걸릴 수 있습니다.")

OFFLOAD_DIR = Path("/content/model_offload")
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(
    str(adapter_dir),
    token=HF_TOKEN,
    trust_remote_code=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
    offload_folder=str(OFFLOAD_DIR),
    offload_state_dict=True,
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
    token=HF_TOKEN,
    device_map="auto",
    offload_folder=str(OFFLOAD_DIR),
)
model = model.merge_and_unload()
model.config.use_cache = True

reset_dir(MERGED_DIR)
model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
    max_shard_size="4GB",
)
tokenizer.save_pretrained(str(MERGED_DIR))

print("Merged HF model saved:", MERGED_DIR)
for path in sorted(MERGED_DIR.iterdir()):
    if path.is_file():
        print("-", path.name, f"({path.stat().st_size / 1024**3:.2f} GB)")

del model, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. llama.cpp 설치 및 빌드

최신 `llama.cpp`를 받아 Hugging Face -> GGUF 변환 스크립트와 `llama-quantize` 바이너리를 준비합니다. `llama.cpp`의 변환 requirements는 일부 버전에서 `transformers` v4를 pin하므로 Gemma 4 변환에서는 직접 설치하지 않습니다.

In [ ]:
LLAMA_CPP_DIR = Path("/content/llama.cpp")
if LLAMA_CPP_DIR.exists():
    shutil.rmtree(LLAMA_CPP_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=5.5.0",
        "gguf",
        "numpy",
        "sentencepiece>=0.1.98,<0.3.0",
        "protobuf>=4.21.0,<6.0.0",
        "safetensors",
        "packaging",
    ],
    check=True,
)

build_dir = LLAMA_CPP_DIR / "build"
subprocess.run(
    ["cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(build_dir), "-DGGML_CUDA=OFF", "-DCMAKE_BUILD_TYPE=Release"],
    check=True,
)
subprocess.run(
    ["cmake", "--build", str(build_dir), "--config", "Release", "-j", "2", "--target", "llama-quantize"],
    check=True,
)

quantize_candidates = [
    build_dir / "bin" / "llama-quantize",
    build_dir / "tools" / "quantize" / "llama-quantize",
    build_dir / "llama-quantize",
]
QUANTIZE_BIN = next((path for path in quantize_candidates if path.exists()), None)
if QUANTIZE_BIN is None:
    raise FileNotFoundError("llama-quantize 바이너리를 찾지 못했습니다.")

print("llama.cpp:", LLAMA_CPP_DIR)
print("quantize:", QUANTIZE_BIN)

import transformers
print("Transformers:", transformers.__version__)


## 6. GGUF 변환 및 Q4_K_M 양자화

먼저 F16 GGUF를 만들고, 그 결과물을 `Q4_K_M`으로 양자화합니다.

In [ ]:
def run_streamed(command, label: str):
    print("\n$", " ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        recent_lines.append(line)
        if len(recent_lines) > 120:
            recent_lines = recent_lines[-120:]
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(recent_lines[-80:])
        raise RuntimeError(f"{label} failed with exit code {return_code}. Recent output:\n{tail}")


def require_transformers_v5_for_gemma4():
    import transformers

    config_path = MERGED_DIR / "config.json"
    config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}
    text_config = config.get("text_config") if isinstance(config.get("text_config"), dict) else {}
    model_type = str(config.get("model_type") or text_config.get("model_type") or "").lower()
    is_gemma4 = "gemma4" in model_type or "gemma-4" in MODEL_ID.lower()
    version = transformers.__version__
    major = int(version.split(".", 1)[0])
    print("Transformers for conversion:", version)
    print("Merged model_type:", model_type or "unknown")
    if is_gemma4 and major < 5:
        raise RuntimeError(
            "Gemma 4 conversion requires Transformers v5+. "
            f"Current version is {version}. Re-run the install and llama.cpp setup cells."
        )


require_transformers_v5_for_gemma4()
reset_dir(GGUF_DIR)

F16_GGUF = GGUF_DIR / "gemma4_news_analyzer.F16.gguf"
Q4_GGUF = GGUF_DIR / "gemma4_news_analyzer.Q4_K_M.gguf"
convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"

run_streamed(
    [
        sys.executable,
        str(convert_script),
        str(MERGED_DIR),
        "--outfile",
        str(F16_GGUF),
        "--outtype",
        "f16",
        "--verbose",
    ],
    "HF -> F16 GGUF conversion",
)

run_streamed(
    [str(QUANTIZE_BIN), str(F16_GGUF), str(Q4_GGUF), "Q4_K_M"],
    "Q4_K_M quantization",
)

print("F16 GGUF:", F16_GGUF, f"{F16_GGUF.stat().st_size / 1024**3:.2f} GB")
print("Q4_K_M GGUF:", Q4_GGUF, f"{Q4_GGUF.stat().st_size / 1024**3:.2f} GB")


## 7. Ollama Modelfile 및 다운로드 패키지 생성

`ollama_package` 폴더에는 `Modelfile`과 최종 `.gguf`만 들어갑니다.

In [ ]:
root_modelfile = OUTPUT_ROOT / "Modelfile"
root_modelfile.write_text(
    f"FROM ./gguf_q4_k_m/{Q4_GGUF.name}\n"
    "TEMPLATE \"{{ .Prompt }}\"\n"
    "PARAMETER temperature 0\n"
    "PARAMETER top_p 0.9\n"
    "PARAMETER top_k 64\n",
    encoding="utf-8",
)

reset_dir(PACKAGE_DIR)
shutil.copy2(Q4_GGUF, PACKAGE_DIR / Q4_GGUF.name)
(PACKAGE_DIR / "Modelfile").write_text(
    f"FROM ./{Q4_GGUF.name}\n"
    "TEMPLATE \"{{ .Prompt }}\"\n"
    "PARAMETER temperature 0\n"
    "PARAMETER top_p 0.9\n"
    "PARAMETER top_k 64\n",
    encoding="utf-8",
)

archive_base = DRIVE_ROOT / "gemma4_news_analyzer_gguf_q4_k_m"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(PACKAGE_DIR))

print("Package directory:", PACKAGE_DIR)
print("Archive:", archive_path)
print("Modelfile:")
print((PACKAGE_DIR / "Modelfile").read_text(encoding="utf-8"))
print("Local Ollama command after unzip:")
print("ollama create gemma4-news-analyzer:q4_k_m -f Modelfile")
print("ollama run gemma4-news-analyzer:q4_k_m")


## 8. 다운로드

브라우저 다운로드가 불안정하면 Drive의 `/MyDrive/dev-jinro/gemma4_news_analyzer_gguf_q4_k_m.zip`에서 직접 받으세요.

In [ ]:
files.download(archive_path)
